# 05 · Review types

Groups reviews into topics (review types) from their `04` embeddings with BERTopic (UMAP + HDBSCAN).

- **`MODE = "fit"`** builds the topics on a sample balanced by game × language × 👍/👎, then assigns every review. It replaces all topics in the database, so labels from `06` must be redone after a refit.
- **`MODE = "assign"`** keeps the topics and only assigns reviews that have no topic yet (after new `01` → `02` → `04` runs).

A review outside the sample gets the most common cluster among its `NEIGHBORS` most similar sample reviews. Special topics: **`-2`** = too short to have a topic ("gg", "好玩", "10/10"), **`-1`** = noise (no clear topic); a noise review is moved to the nearest topic when it is similar enough (`OUTLIER_THRESHOLD`).

In [ ]:
import json
import re
import shutil

import jieba
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from tqdm.auto import tqdm
from umap import UMAP

from src.clean.rules import HAN
from src.config import ROOT, load_games
from src.db import read, write
from src.db.connection import init_db

init_db()
jieba.setLogLevel("WARNING")

In [ ]:
MODE = "fit"                # "fit" = build topics (replaces all), "assign" = only reviews without a topic
SAMPLE_SIZE = 150_000       # reviews to fit on
MIN_CLUSTER_SIZE = 400      # bigger -> fewer, broader topics (400 gave ~65 topics on 3.3M reviews)
MIN_SAMPLES = 5             # bigger -> more reviews left as noise (-1)
OUTLIER_THRESHOLD = 0.70    # noise -> nearest topic if cosine similarity to its center is at least this; 1 = off
NEIGHBORS = 15              # sample reviews that vote on the topic of every other review
SHORT_CHARS = 20            # shorter reviews -> topic -2 (Chinese: fewer than SHORT_HAN characters)
SHORT_HAN = 4
SEED = 0

EMB_DIR = ROOT / "data" / "embeddings" / "BAAI__bge-m3"
MODEL_DIR = ROOT / "data" / "models" / "bertopic"
SHORT, NOISE = -2, -1

## Reviews and embeddings

In [ ]:
# one pass over all kept reviews in chunks: texts are only needed to mark short reviews, then dropped
chunks = []
for chunk in tqdm(read.iter_reviews(columns=["recommendation_id", "appid", "voted_up", "review"],
                                    only_kept=True, flag_columns=["text_language"]), desc="reviews", unit="chunk"):
    text = chunk.pop("review").fillna("").str.strip()
    chunk["short"] = np.where(chunk["text_language"].eq("zh"), text.str.count(HAN) < SHORT_HAN,
                              text.str.len() < SHORT_CHARS)
    chunks.append(chunk)
reviews = pd.concat(chunks, ignore_index=True)
del chunks
reviews["game"] = reviews["appid"].map({g.appid: g.name for g in load_games()})

parts = sorted(EMB_DIR.glob("part-*.parquet"))
index = pd.concat(
    [pd.read_parquet(p, columns=["recommendation_id", "updated_at"]).assign(part=i) for i, p in enumerate(parts)],
    ignore_index=True,
)
index["row"] = index.groupby("part").cumcount()
index = index.sort_values("updated_at").drop_duplicates("recommendation_id", keep="last")


def read_vectors(ids):
    """(ids, float32 vectors) per embeddings file, only for the given review IDs."""
    rows = index[index["recommendation_id"].isin(ids)]
    for i, g in rows.groupby("part"):
        t = pq.read_table(parts[i], columns=["recommendation_id", "embedding"]).take(np.sort(g["row"].to_numpy()))
        vectors = t["embedding"].combine_chunks().flatten().to_numpy().reshape(len(t), -1)
        yield t["recommendation_id"].to_numpy(), vectors.astype(np.float32)


missing = ~reviews["recommendation_id"].isin(index["recommendation_id"])
print(f"{len(reviews):,} kept reviews, {reviews['short'].mean():.0%} short, "
      f"{missing.sum():,} without embeddings (run 04 first; they are skipped)")
reviews = reviews[~missing]

## Keywords

Words that set a topic apart (c-TF-IDF): jieba splits Chinese into words, very common words (game, игра, 游戏) are pushed down. Used for topic keywords and naming, not for the clustering itself.

In [ ]:
WORDS = re.compile(r"[\u3400-\u9fff]+|[^\W\d_]{2,}")
STOP_RU = """и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было
вот от меня еще нет о из ему теперь когда даже ну ли если уже или ни быть был него до вас нибудь опять уж вам
ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз
тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее
сейчас были куда зачем всех никогда можно при об другой хоть после над больше тот через эти нас про всего них
какая много эту моя свою этой перед иногда лучше чуть том такой им более всегда конечно всю между это очень
просто вообще""".split()
STOP_ZH = """的 了 是 我 你 他 她 它 们 这 那 就 都 也 和 有 在 不 很 吧 啊 呢 吗 还 又 被 把 给 让 一个 没有 什么
自己 可以 但是 因为 所以 如果 就是 还是 然后 一下 真的 这个 那个 已经""".split()
STOP_GAME = "game games play игра игру игры игре играть 游戏 这个游戏".split()
STOP = sorted(set(ENGLISH_STOP_WORDS) | set(STOP_RU) | set(STOP_ZH) | set(STOP_GAME))


def tokenize(doc):
    tokens = []
    for m in WORDS.findall(doc.lower()):
        tokens += [w for w in jieba.lcut(m) if w.strip()] if HAN.match(m) else [m]
    return tokens

## Topics: fit (`MODE = "fit"`) or load (`MODE = "assign"`)

In [ ]:
def stratified_sample(pool, keys, size, seed):
    """The same number of reviews from every keys-group; small groups are taken whole."""
    sizes = sorted(pool.groupby(keys).size())
    budget, cap = size, sizes[-1]
    for i, n in enumerate(sizes):
        cap = budget // (len(sizes) - i)
        if n > cap:
            break
        budget -= n
    return pool.sample(frac=1, random_state=seed).groupby(keys).head(cap), cap


def reassign_noise(topics, vectors):
    """Noise reviews -> the topic with the most similar center, if similar enough."""
    topics = np.array(topics)
    noise = topics == NOISE
    if noise.any():
        sims = vectors[noise] @ centroids.T
        topics[noise] = np.where(sims.max(axis=1) >= OUTLIER_THRESHOLD, sims.argmax(axis=1), NOISE)
    return topics


if MODE == "fit":
    pool = reviews[~reviews["short"] & reviews["text_language"].isin(["en", "ru", "zh"])]
    sample, cap = stratified_sample(pool, ["appid", "text_language", "voted_up"], SAMPLE_SIZE, SEED)
    texts = read.reviews(ids=sample["recommendation_id"], columns=["recommendation_id", "review"], with_flags=False)
    sample = sample.merge(texts, on="recommendation_id")
    ids, vectors = map(np.concatenate, zip(*read_vectors(sample["recommendation_id"])))
    sample = sample.set_index("recommendation_id").loc[ids].reset_index()
    print(f"sample: {len(sample):,} reviews, up to {cap:,} per game × language × 👍/👎")

    topic_model = BERTopic(
        language="multilingual",  # the default "english" strips every non-Latin character from keywords
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED),
        hdbscan_model=HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES),
        vectorizer_model=CountVectorizer(
            tokenizer=tokenize, token_pattern=None, lowercase=False, stop_words=STOP, ngram_range=(1, 2)
        ),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        verbose=False,
    )
    sample["cluster"] = topic_model.fit_transform(sample["review"].tolist(), vectors)[0]
    clusters = sample["cluster"].to_numpy()
    centroids = np.stack([vectors[clusters == t].mean(axis=0) for t in range(clusters.max() + 1)])
    centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
    sample["topic"] = reassign_noise(clusters, vectors)
    topic_words = pd.DataFrame(
        [(t, rank, word, score) for t, words in topic_model.get_topics().items() if t >= 0
         for rank, (word, score) in enumerate(words)],
        columns=["topic", "rank", "word", "score"],
    )

    # 2D map coordinates for 30k sample reviews (charts in 06)
    pick = np.random.default_rng(SEED).choice(len(sample), min(30_000, len(sample)), replace=False)
    xy = UMAP(n_components=2, min_dist=0.1, random_state=SEED).fit_transform(topic_model.umap_model.embedding_[pick])
    sample[["x", "y"]] = np.nan
    sample.iloc[pick, sample.columns.get_indexer(["x", "y"])] = xy
    del topic_model
    print(f"{len(centroids)} topics; noise in the sample {np.mean(clusters == NOISE):.0%}, "
          f"{sample['topic'].eq(NOISE).mean():.0%} after moving similar noise to topics")
else:
    sample = pd.read_parquet(MODEL_DIR / "sample.parquet")
    centroids = np.load(MODEL_DIR / "centroids.npy")
    topic_words = pd.read_parquet(MODEL_DIR / "topic_words.parquet")
    ids, vectors = map(np.concatenate, zip(*read_vectors(sample["recommendation_id"])))
    sample = sample.set_index("recommendation_id").loc[ids].reset_index()
    print(f"loaded {len(centroids)} topics fitted on {len(sample):,} reviews")

keywords = topic_words[topic_words["rank"] < 10].groupby("topic")["word"].agg(", ".join).to_dict()

### Checks (fit)

Aim for roughly 30–80 topics. Too many → raise `MIN_CLUSTER_SIZE`; too much noise → lower `MIN_SAMPLES` or `OUTLIER_THRESHOLD`. A topic that is >90% one language is often fine (a game mostly one language reviews, slang) or a duplicate of a topic in another language: those get the same label in `06`. Maps and the topic tree are in `06`.

In [ ]:
if MODE == "fit":
    def top_share(s):
        shares = s.value_counts(normalize=True)
        return f"{shares.index[0]} {shares.iloc[0]:.0%}"

    clustered = sample[sample["topic"] != NOISE]
    topics_view = clustered.groupby("topic").agg(
        size=("topic", "size"),
        language=("text_language", top_share),
        game=("game", top_share),
        positive=("voted_up", "mean"),
    )
    topics_view["keywords"] = topics_view.index.map(lambda t: ", ".join(keywords[t].split(", ")[:8]))
    one_language = clustered.groupby("topic")["text_language"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
    print(f"topics that are >90% one language: {(one_language > 0.9).mean():.0%}")
    with pd.option_context("display.max_rows", 200, "display.max_colwidth", 90):
        display(topics_view.round({"positive": 2}))

In [ ]:
if MODE == "fit":
    noise = sample.assign(noise=sample["topic"].eq(NOISE))
    display(noise.pivot_table(index="game", columns="text_language", values="noise", aggfunc="mean")
            .style.format("{:.0%}").set_caption("noise share in the sample"))

## Assign

`fit`: every review outside the sample. `assign`: only reviews without a topic. One embeddings file at a time; short reviews get `-2` without a vote.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
reference = torch.from_numpy(vectors).to(device, torch.float16 if device == "cuda" else torch.float32)
reference_clusters = torch.from_numpy(sample["cluster"].to_numpy()).to(device)
del vectors


def nearest_cluster(vectors, batch=4096):
    """Most common cluster (noise included) among the NEIGHBORS most similar sample reviews."""
    out = []
    for b in range(0, len(vectors), batch):
        query = torch.from_numpy(vectors[b : b + batch]).to(device, reference.dtype)
        neighbors = (query @ reference.T).topk(NEIGHBORS, dim=1).indices
        out.append(torch.mode(reference_clusters[neighbors], dim=1).values.cpu().numpy())
    return np.concatenate(out)


if MODE == "fit":
    done = sample["recommendation_id"]
else:
    done = read.reviews(columns=["recommendation_id"], with_flags=False, with_topics=True)
    done = done.loc[done["topic_id"].notna(), "recommendation_id"]
todo = reviews[~reviews["recommendation_id"].isin(done)]
to_vote = todo.loc[~todo["short"], "recommendation_id"]

assigned = [pd.DataFrame({"recommendation_id": todo.loc[todo["short"], "recommendation_id"], "topic_id": SHORT})]
with tqdm(total=len(to_vote), unit="review") as bar:
    for ids, vectors in read_vectors(to_vote):
        topics = reassign_noise(nearest_cluster(vectors), vectors)
        assigned.append(pd.DataFrame({"recommendation_id": ids, "topic_id": topics}))
        bar.update(len(ids))
if MODE == "fit":
    assigned.append(sample[["recommendation_id", "topic"]].rename(columns={"topic": "topic_id"}))
assigned = pd.concat(assigned, ignore_index=True)
print(f"{len(assigned):,} reviews assigned")

In [ ]:
new = todo.merge(assigned, on="recommendation_id")
summary = new.groupby("game").agg(
    reviews=("topic_id", "size"),
    short=("topic_id", lambda t: (t == SHORT).mean()),
    noise=("topic_id", lambda t: (t == NOISE).mean()),
)
summary.style.format({"short": "{:.0%}", "noise": "{:.0%}"}).set_caption(
    "assigned now: a game with much more noise than the others is not covered by the topics -> refit"
)

## Save

In [ ]:
if MODE == "fit":
    if MODEL_DIR.exists():
        shutil.rmtree(MODEL_DIR)
    MODEL_DIR.mkdir(parents=True)
    columns = ["recommendation_id", "cluster", "topic", "x", "y", "appid", "game", "text_language", "voted_up", "review"]
    sample[columns].to_parquet(MODEL_DIR / "sample.parquet")
    np.save(MODEL_DIR / "centroids.npy", centroids)
    topic_words.to_parquet(MODEL_DIR / "topic_words.parquet")
    params = dict(SAMPLE_SIZE=SAMPLE_SIZE, MIN_CLUSTER_SIZE=MIN_CLUSTER_SIZE, MIN_SAMPLES=MIN_SAMPLES,
                  OUTLIER_THRESHOLD=OUTLIER_THRESHOLD, NEIGHBORS=NEIGHBORS, SHORT_CHARS=SHORT_CHARS,
                  SHORT_HAN=SHORT_HAN, SEED=SEED, fitted_at=pd.Timestamp.now().isoformat(timespec="seconds"))
    (MODEL_DIR / "params.json").write_text(json.dumps(params, indent=2))

    sizes = assigned["topic_id"].value_counts()
    topics = pd.DataFrame({"topic_id": sorted({SHORT, NOISE, *keywords})})
    topics["n_reviews"] = topics["topic_id"].map(sizes).fillna(0).astype(int)
    topics["keywords"] = topics["topic_id"].map(keywords)
    topics["label"] = None
    write.replace_topics(topics, assigned)
else:
    write.add_review_topics(assigned)
read.topics()

## Material for naming (`06`)

`topics_for_naming.md`: per topic its keywords, language / game / 👍 mix, its most similar topics (likely the same theme in another language → same label) and 10 example reviews across languages.

In [ ]:
if MODE == "fit":
    similar = centroids @ centroids.T
    np.fill_diagonal(similar, -1)
    lines = ["# Topics for naming", "", f"{len(keywords)} topics from a fit on {len(sample):,} reviews.", ""]
    for t in sorted(keywords):
        members = sample[sample["topic"] == t]
        examples = members.sample(frac=1, random_state=SEED).groupby("text_language").head(4).head(10)
        langs = ", ".join(f"{k} {v:.0%}" for k, v in members["text_language"].value_counts(normalize=True).items())
        games = ", ".join(f"{k} {v:.0%}" for k, v in members["game"].value_counts(normalize=True).head(3).items())
        lines += [
            f"## Topic {t}", "",
            f"- size in sample: {len(members):,}; positive: {members['voted_up'].mean():.0%}",
            f"- languages: {langs}", f"- games: {games}", f"- keywords: {keywords[t]}",
            "- most similar topics: " + ", ".join(f"{o} ({similar[t, o]:.2f})" for o in similar[t].argsort()[::-1][:3]),
            "", "Examples:", "",
        ]
        lines += [f"- [{r.text_language}] " + " ".join(r.review.split())[:300] for r in examples.itertuples()]
        lines.append("")
    (MODEL_DIR / "topics_for_naming.md").write_text("\n".join(lines), encoding="utf-8")
    print(MODEL_DIR / "topics_for_naming.md")